# Proyecto #2: Maze Solver (v2)

Esta versión sigue la estructura de funciones y el estilo del laboratorio `local_search_lab.ipynb`, adaptado para los algoritmos de búsqueda informada y no informada solicitados.

In [1]:
import time
import math
import heapq
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

# --- Estructura de Nodo (Basada en la convención de laboratorios de búsqueda) ---
class Nodo:
    def __init__(self, estado, padre=None, g=0, h=0):
        self.estado = estado
        self.padre = padre
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, otro):
        return self.f < otro.f

# --- Funciones Útiles (Inspiradas en local_search_lab.ipynb) ---

def cargar_laberinto(file_path):
    with open(file_path, 'r') as f:
        grid = [list(line.strip()) for line in f.readlines()]
    return grid

def encontrar_valor(grid, val):
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] == val:
                return (r, c)
    return None

def es_meta(estado, meta):
    return estado == meta

def heuristica(estado, meta, tipo='manhattan'):
    if tipo == 'manhattan':
        return abs(estado[0] - meta[0]) + abs(estado[1] - meta[1])
    elif tipo == 'euclidiana':
        return math.sqrt((estado[0] - meta[0])**2 + (estado[1] - meta[1])**2)
    return 0

def generar_sucesores(estado, grid):
    sucesores = []
    rows, cols = len(grid), len(grid[0])
    # Jerarquía: Arriba, Derecha, Abajo, Izquierda
    movimientos = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    
    for dr, dc in movimientos:
        nr, nc = estado[0] + dr, estado[1] + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '1':
            sucesores.append((nr, nc))
    return sucesores

def reconstruir_camino(nodo):
    camino = []
    actual = nodo
    while actual:
        camino.append(actual.estado)
        actual = actual.padre
    return camino[::-1]

def calcular_branching_factor(nodos_explorados, profundidad):
    if profundidad == 0 or nodos_explorados <= 1:
        return 0
    return nodos_explorados ** (1/profundidad)

# --- Algoritmos de Búsqueda ---

def bfs(grid, inicio, meta):
    tiempo_inicio = time.time()
    frontera = deque([Nodo(inicio)])
    visitados = {inicio}
    nodos_explorados = 0
    
    while frontera:
        nodo_actual = frontera.popleft()
        nodos_explorados += 1
        
        if es_meta(nodo_actual.estado, meta):
            return {
                'solucion': True,
                'camino': reconstruir_camino(nodo_actual),
                'nodos': nodos_explorados,
                'tiempo': time.time() - tiempo_inicio
            }
        
        for sucesor in generar_sucesores(nodo_actual.estado, grid):
            if sucesor not in visitados:
                visitados.add(sucesor)
                frontera.append(Nodo(sucesor, nodo_actual))
                
    return {'solucion': False, 'nodos': nodos_explorados, 'tiempo': time.time() - tiempo_inicio}

def dfs(grid, inicio, meta):
    tiempo_inicio = time.time()
    frontera = [Nodo(inicio)]
    visitados = {inicio}
    nodos_explorados = 0
    
    while frontera:
        nodo_actual = frontera.pop()
        nodos_explorados += 1
        
        if es_meta(nodo_actual.estado, meta):
            return {
                'solucion': True,
                'camino': reconstruir_camino(nodo_actual),
                'nodos': nodos_explorados,
                'tiempo': time.time() - tiempo_inicio
            }
        
        # Inverso para mantener prioridad Arriba, Derecha, Abajo, Izquierda
        sucesores = generar_sucesores(nodo_actual.estado, grid)
        for sucesor in reversed(sucesores):
            if sucesor not in visitados:
                visitados.add(sucesor)
                frontera.append(Nodo(sucesor, nodo_actual))
                
    return {'solucion': False, 'nodos': nodos_explorados, 'tiempo': time.time() - tiempo_inicio}

def greedy(grid, inicio, meta, h_tipo='manhattan'):
    tiempo_inicio = time.time()
    h_inic = heuristica(inicio, meta, h_tipo)
    frontera = [(h_inic, Nodo(inicio, h=h_inic))]
    visitados = {inicio}
    nodos_explorados = 0
    
    while frontera:
        _, nodo_actual = heapq.heappop(frontera)
        nodos_explorados += 1
        
        if es_meta(nodo_actual.estado, meta):
            return {
                'solucion': True,
                'camino': reconstruir_camino(nodo_actual),
                'nodos': nodos_explorados,
                'tiempo': time.time() - tiempo_inicio
            }
        
        for sucesor in generar_sucesores(nodo_actual.estado, grid):
            if sucesor not in visitados:
                visitados.add(sucesor)
                h = heuristica(sucesor, meta, h_tipo)
                heapq.heappush(frontera, (h, Nodo(sucesor, nodo_actual, h=h)))
                
    return {'solucion': False, 'nodos': nodos_explorados, 'tiempo': time.time() - tiempo_inicio}

def a_star(grid, inicio, meta, h_tipo='manhattan'):
    tiempo_inicio = time.time()
    h_inic = heuristica(inicio, meta, h_tipo)
    # (f, Nodo)
    frontera = [(h_inic, Nodo(inicio, h=h_inic))]
    g_scores = {inicio: 0}
    nodos_explorados = 0
    
    while frontera:
        f_val, nodo_actual = heapq.heappop(frontera)
        nodos_explorados += 1
        
        if es_meta(nodo_actual.estado, meta):
            return {
                'solucion': True,
                'camino': reconstruir_camino(nodo_actual),
                'nodos': nodos_explorados,
                'tiempo': time.time() - tiempo_inicio
            }
        
        for sucesor in generar_sucesores(nodo_actual.estado, grid):
            nuevo_g = nodo_actual.g + 1
            if sucesor not in g_scores or nuevo_g < g_scores[sucesor]:
                g_scores[sucesor] = nuevo_g
                h = heuristica(sucesor, meta, h_tipo)
                nuevo_nodo = Nodo(sucesor, nodo_actual, g=nuevo_g, h=h)
                heapq.heappush(frontera, (nuevo_nodo.f, nuevo_nodo))
                
    return {'solucion': False, 'nodos': nodos_explorados, 'tiempo': time.time() - tiempo_inicio}

# --- Visualización y Experimentos ---

def visualizar_laberinto(grid, camino=None, titulo="Laberinto"):
    rows, cols = len(grid), len(grid[0])
    data = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1': data[r, c] = 0
            elif grid[r][c] == '2': data[r, c] = 0.5
            elif grid[r][c] == '3': data[r, c] = 0.7
            else: data[r, c] = 1
    
    plt.figure(figsize=(7, 7))
    plt.imshow(data, cmap='gray')
    if camino:
        rs, cs = zip(*camino)
        plt.plot(cs, rs, color='red', linewidth=1)
    plt.title(titulo)
    plt.axis('off')
    plt.show()

def ejecutar_comparativa(archivo_maze):
    grid = cargar_laberinto(archivo_maze)
    inicio = encontrar_valor(grid, '2')
    meta = encontrar_valor(grid, '3')
    
    algos = [
        ("BFS", bfs, {}),
        ("DFS", dfs, {}),
        ("Greedy (Manhattan)", greedy, {'h_tipo': 'manhattan'}),
        ("Greedy (Euclidiana)", greedy, {'h_tipo': 'euclidiana'}),
        ("A* (Manhattan)", a_star, {'h_tipo': 'manhattan'}),
        ("A* (Euclidiana)", a_star, {'h_tipo': 'euclidiana'})
    ]

    print(f"{ 'Algoritmo':<20 } | { 'Longitud':<10 } | { 'Nodos':<10 } | { 'Tiempo (s)':<12 } | { 'B-Factor':<10 }")
    print("-" * 70)

    for nombre, func, kwargs in algos:
        res = func(grid, inicio, meta, **kwargs)
        if res['solucion']:
            length = len(res['camino'])
            bf = calcular_branching_factor(res['nodos'], length)
            print(f"{ nombre:<20 } | { length:<10 } | { res['nodos']:<10 } | { res['tiempo']:<12.6f } | { bf:<10.4f }")
            visualizar_laberinto(grid, res['camino'], titulo=nombre)
        else:
            print(f"{ nombre:<20 } | No se encontró solución")

ejecutar_comparativa('test_maze.txt')

ValueError: Unknown format code '\x20' for object of type 'str'